[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megacare-dev/agentic_rag_workshop/blob/main/th/4hr/agentic_rag_4hr_homework.ipynb)

# 📝 แบบฝึกหัด: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 ติดตั้ง Dependencies

In [2]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'scikit-learn']:
    _pip_install(_pkg)

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

  ⏭️  google-adk: skipped
  ⏭️  google-genai: skipped
  ⏭️  sentence-transformers: skipped
  ✅ qdrant-client: done
  ✅ langchain-text-splitters: done
  ⏭️  scikit-learn: skipped
✅ ติดตั้งเรียบร้อย!
CPU times: user 1.26 s, sys: 175 ms, total: 1.44 s
Wall time: 16.9 s


## 🎓 กรอกข้อมูลนักศึกษา

In [1]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = 'จิรพัฒน์ บำรุงกุล'   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = '673040378-0'   # เช่น '6512345678'
PHONE        = '062-345-5980'   # เช่น '081-234-5678'
LINE_ID      = 'dn032x'   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

✅ ชื่อ-นามสกุล: จิรพัฒน์ บำรุงกุล
✅ รหัสนักศึกษา: 673040378-0
📱 เบอร์โทร: 062-345-5980
💬 LINE ID: dn032x


## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [3]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

✅ สร้างข้อมูลเฉพาะสำหรับ 673040378-0
📁 จำนวนไฟล์: 8 ไฟล์
🔍 Query เฉพาะตัว: "เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ"
  📄 doc_1.txt (120 ตัวอักษร)
  📄 doc_2.txt (121 ตัวอักษร)
  📄 doc_3.txt (150 ตัวอักษร)
  📄 doc_4.txt (141 ตัวอักษร)
  📄 doc_5.txt (146 ตัวอักษร)
  📄 doc_6.txt (146 ตัวอักษร)
  📄 doc_7.txt (136 ตัวอักษร)
  📄 doc_8.txt (146 ตัวอักษร)
CPU times: user 1.76 ms, sys: 0 ns, total: 1.76 ms
Wall time: 1.79 ms


---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน)

- รวมข้อความจากทุกไฟล์ใน `homework_data/`
- Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
- ค้นหาด้วย query: `MY_QUERY` (ที่สร้างจากรหัสนักศึกษา)
- เก็บลง Qdrant collection ชื่อ `f'hw_{STUDENT_ID}'`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ไหนมี similarity สูงสุด? (score ทศนิยม 4 ตำแหน่ง)
3. Top-3 ผลลัพธ์จาก Qdrant มี score เท่าไร?

In [4]:
# ขั้นตอนที่ 1: เขียนโค้ดที่นี่

# 💡 Hint:
#   1. อ่านไฟล์จาก 'homework_data/' รวมเป็น text เดียว
#   2. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   3. splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
#   4. chunks = splitter.split_text(all_text)
#   5. from sentence_transformers import SentenceTransformer
#   6. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   7. passages = ['passage: ' + c for c in chunks]
#   8. embeddings = model.encode(passages)
#   9. query_emb = model.encode(f'query: {MY_QUERY}')
#  10. ใช้ cosine_similarity() หา chunk ที่คล้ายสุด
#  11. from qdrant_client import QdrantClient, models
#  12. สร้าง collection, upsert, query_points
# ขั้นตอนที่ 1: Chunk + Embed + Search

import os
import glob
import numpy as np

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# ============================================================
# 1. อ่านไฟล์ทั้งหมดจาก homework_data/
# ============================================================

all_text = ""

files = sorted(glob.glob("homework_data/*.txt"))

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
        all_text += text + "\n"

print(f"📁 อ่านไฟล์ทั้งหมด: {len(files)} ไฟล์")


# ============================================================
# 2. Chunk ข้อความ
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

chunks = splitter.split_text(all_text)

print(f"📦 จำนวน chunks ทั้งหมด: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}: {chunk}")


# ============================================================
# 3. สร้าง Embedding
# ============================================================

model = SentenceTransformer("intfloat/multilingual-e5-large")

# E5 แนะนำให้ใส่ prefix passage:
passages = ["passage: " + c for c in chunks]

embeddings = model.encode(
    passages,
    normalize_embeddings=True
)

print(f"\n🔢 Embedding shape: {embeddings.shape}")


# ============================================================
# 4. สร้าง Query Embedding
# ============================================================

query_emb = model.encode(
    "query: " + MY_QUERY,
    normalize_embeddings=True
)

print(f"🔍 Query: {MY_QUERY}")


# ============================================================
# 5. หา Cosine Similarity
# ============================================================

similarities = cosine_similarity(
    [query_emb],
    embeddings
)[0]

sorted_indices = np.argsort(similarities)[::-1]

print("\n🏆 Top similarity จาก cosine similarity:")

for rank, idx in enumerate(sorted_indices[:3], start=1):
    print(
        f"{rank}. Chunk {idx} "
        f"score={similarities[idx]:.4f}"
    )
    print(f"   {chunks[idx]}")


# ============================================================
# 6. สร้าง Qdrant Collection
# ============================================================

collection_name = f"hw_{STUDENT_ID}"

qdrant = QdrantClient(":memory:")

# ขนาด vector จาก model
vector_size = embeddings.shape[1]

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE
    )
)

print(f"\n🗄️ Qdrant collection: {collection_name}")


# ============================================================
# 7. เตรียมข้อมูลสำหรับ Qdrant
# ============================================================

points = []

for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):

    points.append(
        models.PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": chunk,
                "chunk_id": i
            }
        )
    )


# ============================================================
# 8. Upsert ลง Qdrant
# ============================================================

qdrant.upsert(
    collection_name=collection_name,
    points=points
)

print(f"✅ Upsert {len(points)} vectors สำเร็จ")


# ============================================================
# 9. Search จาก Qdrant
# ============================================================

qdrant_results = qdrant.query_points(
    collection_name=collection_name,
    query=query_emb.tolist(),
    limit=3,
    with_payload=True
).points


# ============================================================
# 10. แสดงผล Top-3
# ============================================================

print("\n🔎 Qdrant Top-3 Results:")

for rank, result in enumerate(qdrant_results, start=1):

    print(
        f"\n{rank}. "
        f"Chunk {result.payload['chunk_id']} "
        f"score={result.score:.4f}"
    )

    print(result.payload["text"])


# ============================================================
# Self-check
# ============================================================

assert len(chunks) > 0, "❌ ยังไม่ได้ chunk"
assert len(qdrant_results) == 3, "❌ ควรได้ top_k=3 จาก Qdrant"

print(
    f"\n✅ Step 1 passed: "
    f"{len(chunks)} chunks, "
    f"top score={qdrant_results[0].score:.4f}"
)



# ✅ Self-check (uncomment หลังเขียนโค้ดเสร็จ)
# assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
# assert len(qdrant_results) == 3, '❌ ควรได้ top_k=3 จาก Qdrant'
# print(f'✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

📁 อ่านไฟล์ทั้งหมด: 8 ไฟล์
📦 จำนวน chunks ทั้งหมด: 9

Chunk 0: Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้

Chunk 1: Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini

Chunk 2: Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM

Chunk 3: LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้

Chunk 4: Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว

Chunk 5: Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ

Chunk 6: Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี

Chunk 7: Chunking คือกระบวนการแบ่งเอกสารขนา

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]


🔢 Embedding shape: (9, 1024)
🔍 Query: เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ

🏆 Top similarity จาก cosine similarity:
1. Chunk 3 score=0.8247
   LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้
2. Chunk 2 score=0.8189
   Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM
3. Chunk 4 score=0.7948
   Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว

🗄️ Qdrant collection: hw_673040378-0
✅ Upsert 9 vectors สำเร็จ

🔎 Qdrant Top-3 Results:

1. Chunk 3 score=0.8247
LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้

2. Chunk 2 score=0.8189
Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM

3. Chunk 4 score=0.7948
Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว

✅ Step 1 passed: 9 chunks, top score=0.8247

---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน)

- ตั้งค่า Gemini API Key (Colab Secrets)
- สร้าง **Custom Tool** อย่างน้อย 1 ตัว (ห้ามซ้ำกับ BMI ในคาบ)
- สร้าง **Agent** ด้วย Google ADK ที่ใช้ Tool ได้
- ทดลองคุย → แสดงว่า Agent เรียก Tool ได้จริง

**📝 รายงาน:**
1. Tool ของคุณทำอะไร? (อธิบาย 1-2 ประโยค)
2. แสดง output ที่ Agent เรียก Tool สำเร็จ
3. ทำไม docstring ถึงสำคัญ? (อธิบาย 1-2 ประโยค)

In [15]:
# ขั้นตอนที่ 2: Agent + Custom Tool

# ============================================================
# ตั้งค่า API Key
# ============================================================

import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    os.environ["GOOGLE_API_KEY"] = input("🔑 วาง API Key: ")


# ============================================================
# Import Google ADK
# ============================================================

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types


# ============================================================
# Custom Tool
# ============================================================

def calculate_rectangle_area(width: float, length: float) -> str:
    """คำนวณพื้นที่ของสี่เหลี่ยมผืนผ้า

    Args:
        width: ความกว้างของสี่เหลี่ยม หน่วยเป็นเมตร
        length: ความยาวของสี่เหลี่ยม หน่วยเป็นเมตร

    Returns:
        พื้นที่ของสี่เหลี่ยมผืนผ้าในหน่วยตารางเมตร
    """

    area = width * length

    return f"พื้นที่สี่เหลี่ยมผืนผ้า = {area:.2f} ตารางเมตร"


tool = FunctionTool(calculate_rectangle_area)


# ============================================================
# สร้าง Agent
# ============================================================

my_agent = LlmAgent(
    name="area_assistant",
    model="gemini-3.6-flash",
    instruction="""
คุณเป็นผู้ช่วยคำนวณพื้นที่สี่เหลี่ยมผืนผ้า

เมื่อผู้ใช้ถามเกี่ยวกับการคำนวณพื้นที่
ให้เรียกใช้ calculate_rectangle_area tool
ก่อนตอบคำถาม

หลังจากได้รับผลลัพธ์จาก tool แล้ว
ให้อธิบายคำตอบเป็นภาษาไทยอย่างชัดเจน
""",
    tools=[tool]
)


# ============================================================
# Function สำหรับคุยกับ Agent
# ============================================================

async def chat_with_agent(agent, message):

    runner = InMemoryRunner(
        agent=agent,
        app_name="homework"
    )

    session = await runner.session_service.create_session(
        app_name="homework",
        user_id="student"
    )

    content = genai_types.Content(
        role="user",
        parts=[
            genai_types.Part(text=message)
        ]
    )

    response_text = ""

    async for event in runner.run_async(
        user_id="student",
        session_id=session.id,
        new_message=content
    ):

        if event.content and event.content.parts:

            for part in event.content.parts:

                if part.text:
                    response_text += part.text

    return response_text


# ============================================================
# ทดสอบ Agent
# ============================================================

answer = await chat_with_agent(
    my_agent,
    "ห้องสี่เหลี่ยมกว้าง 5 เมตร และยาว 8 เมตร มีพื้นที่เท่าไร?"
)

print(f"🤖 Agent: {answer}")


# ============================================================
# Self-check
# ============================================================

assert answer is not None and len(answer) > 0

print("✅ Step 2 passed!")

🤖 Agent: ห้องสี่เหลี่ยมที่มีความกว้าง 5 เมตร และความยาว 8 เมตร มีพื้นที่ทั้งหมด **40 ตารางเมตร** ครับ

*(คำนวณจากสูตร: พื้นที่ = ความกว้าง × ความยาว = 5 × 8 = 40 ตารางเมตร)*
✅ Step 2 passed!


---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน)

- สร้าง **RAG Tool** ที่ค้นจาก Qdrant (ใช้ collection จากขั้นตอนที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร? (2-3 ประโยค)

In [18]:
# ขั้นตอนที่ 3: RAG Agent + Judge

# ============================================================
# A) RAG Tool
# ============================================================

def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning และ NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา

    Returns:
        ข้อมูลที่เกี่ยวข้องจากฐานความรู้ Top-3
    """

    # สร้าง embedding ของ query
    query_embedding = model.encode(
        "query: " + query,
        normalize_embeddings=True
    )

    # ค้นหาใน Qdrant
    results = qdrant.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        limit=3,
        with_payload=True
    ).points

    # รวมผลลัพธ์เป็นข้อความ
    if not results:
        return "ไม่พบข้อมูลที่เกี่ยวข้อง"

    output = []

    for i, result in enumerate(results, start=1):

        text = result.payload.get("text", "")
        score = result.score

        output.append(
            f"[ผลลัพธ์ที่ {i} | similarity={score:.4f}]\n"
            f"{text}"
        )

    return "\n\n".join(output)


# ============================================================
# B) สร้าง RAG Agent
# ============================================================

rag_agent = LlmAgent(
    name="rag_assistant",
    model="gemini-3.6-flash",
    instruction="""
คุณเป็น AI Assistant ที่ตอบคำถามโดยใช้ฐานความรู้

เมื่อผู้ใช้ถามคำถาม:
1. ต้องเรียก search_knowledge ก่อนเสมอ
2. ใช้ข้อมูลที่ได้จาก tool เป็นหลักในการตอบ
3. ห้ามแต่งข้อมูลที่ไม่มีอยู่ในฐานความรู้
4. ตอบเป็นภาษาไทย
5. อธิบายให้เข้าใจง่ายและกระชับ
""",
    tools=[
        FunctionTool(search_knowledge)
    ]
)


# ============================================================
# C) คำถาม 3 ข้อ
# ============================================================

questions = [
    MY_QUERY,
    "Embedding คืออะไร?",
    "ทำไม RAG ถึงสำคัญ?"
]


# ============================================================
# ถาม RAG Agent
# ============================================================

rag_answers = []

for q in questions:

    ans = await chat_with_agent(
        rag_agent,
        q
    )

    rag_answers.append({
        "question": q,
        "answer": ans
    })

    print("\n" + "=" * 60)
    print(f"❓ {q}")
    print(f"🤖 {ans}")


# ============================================================
# D) LLM-as-Judge
# ============================================================

from google import genai

judge_client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)


JUDGE_PROMPT = """คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}

คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:

- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบเป็น JSON เท่านั้น:

{{"score": 0, "reason": "..."}}
"""


print("\n" + "=" * 60)
print("📊 LLM-as-Judge Results:")


judge_results = []

for qa in rag_answers:

    prompt = JUDGE_PROMPT.format(
        question=qa["question"],
        answer=qa["answer"]
    )

    resp = judge_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    result = json.loads(resp.text)

    judge_results.append({
        "question": qa["question"],
        "score": result["score"],
        "reason": result["reason"]
    })

    print(
        f"❓ {qa['question'][:40]}..."
        f" → ⭐ {result['score']}/5"
        f" — {result['reason']}"
    )


# ============================================================
# Self-check
# ============================================================

assert len(rag_answers) == 3

print(
    "\n✅ Step 3 passed: "
    "ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!"
)


❓ เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ
🤖 เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบคือ **RAG (Retrieval Augmented Generation)** ซึ่งมีรายละเอียดและองค์ประกอบสำคัญ ดังนี้ครับ:

1. **Retrieval Augmented Generation (RAG):** เป็นเทคนิคที่รวม **การค้นหาข้อมูล** เข้ากับ **การสร้างข้อความของโมเดลภาษาขนาดใหญ่ (LLM)** ช่วยให้ได้คำตอบที่ถูกต้อง แม่นยำ และสามารถอ้างอิงแหล่งข้อมูลต้นทางได้
2. **การใช้ Vector Database:** ใช้สำหรับจัดเก็บและค้นหาข้อมูลในรูปแบบ *Embedding Vector* เพื่อค้นหาข้อมูลที่มีความหมายคล้ายคลึงกับคำถามได้อย่างรวดเร็วและมีประสิทธิภาพ
3. **การใช้ Prompt Engineering:** เป็นการออกแบบคำสั่ง (Prompt) นำเอาข้อมูลที่ค้นพบป้อนให้กับ LLM เพื่อควบคุมและปรับปรุงคุณภาพของคำตอบให้ตรงตามความต้องการอย่างถูกต้อง

❓ Embedding คืออะไร?
🤖 **Embedding** (หรือ Embedding Vector) คือ การแปลงข้อมูล (เช่น ข้อความหรือเอกสาร) ให้อยู่ในรูปแบบของชุดตัวเลขเวกเตอร์ (Vector) ที่แสดงถึงความหมายของข้อมูลนั้นๆ

**ประเด็นสำคัญของ Embedding จากฐานข้อมูล:**
* **ช่วยในการค้นหาตามความหมาย:** ข้อมูลที่อยู่ในรูปแบบ Embeddin

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 13.0969932s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, '

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')